Cell:1: Installation

In [ ]:
!pip install -q upet phonopy pymatgen ase "nvalchemi-toolkit-ops==0.3.0"



In [ ]:
import importlib.util, glob, os

spec = importlib.util.find_spec("nvalchemiops")
if spec is None:
    raise SystemExit("nvalchemiops not installed — run: pip install 'nvalchemi-toolkit-ops==0.3.0'")

pkg_dir = os.path.dirname(spec.origin)
hits = glob.glob(os.path.join(pkg_dir, "**", "naive.py"), recursive=True)
print("package at:", pkg_dir)
print("candidates:", hits)

OLD = "    if neighbor_matrix is None:"
NEW = "    max_neighbors = int(max_neighbors)\n    if neighbor_matrix is None:"

for fp in hits:
    src = open(fp).read()
    if "max_neighbors = int(max_neighbors)" in src:
        print("already patched:", fp)
    elif OLD in src:
        open(fp, "w").write(src.replace(OLD, NEW, 1))
        print("patched:", fp)
    else:
        print("anchor line not found:", fp)

CELL 3 — Build the PET-OAM-XL calculator

In [ ]:
from upet.calculator import UPETCalculator

DEVICE = "cuda"   # set "cpu" if no GPU
PET_CALC = UPETCalculator(model="pet-oam-xl", version="1.0.0", device=DEVICE)
print("PET-OAM-XL calculator ready")

CELL 4 — Upload + unzip the seed CIFs and the 2000-candidate CSV

In [ ]:
import zipfile, os, glob, shutil
from google.colab import files

# upload cifs.zip (your 37 seeds) and the 2000-row CSV
print("Upload cifs.zip and batch_1.csv:")
up = files.upload()

# unzip seeds into seeds/
for name in up:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as z: z.extractall("seeds_raw")
os.makedirs("seeds", exist_ok=True)
for f in glob.glob("seeds_raw/**/*.cif", recursive=True):
    shutil.copy(f, os.path.join("seeds", os.path.basename(f)))
print("seed CIFs:", len(glob.glob("seeds/*.cif")))

CSV_PATH = [n for n in up if n.endswith(".csv")][0]
print("candidates CSV:", CSV_PATH)


CELL 5 — Relaxation helpers (npj protocol: FIRE + FrechetCellFilter)

In [ ]:
import numpy as np, json, warnings
warnings.filterwarnings("ignore")
from ase.optimize import FIRE
from ase.filters import FrechetCellFilter
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.vasp import Poscar
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

ADAPTOR = AseAtomsAdaptor()

class NumpyEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.integer): return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return super().default(o)

def relax_atoms(structure, calc, fmax, max_steps=500, relax_cell=True):
    """FIRE + FrechetCellFilter relaxation. Returns relaxed structure, energy,
    convergence info, and symmetry before/after (to preserve PBE space group)."""
    try:
        sg_before = SpacegroupAnalyzer(structure, symprec=1e-3).get_space_group_number()
    except Exception:
        sg_before = None
    atoms = ADAPTOR.get_atoms(structure)
    atoms.calc = calc
    target = FrechetCellFilter(atoms) if relax_cell else atoms
    opt = FIRE(target, logfile="/dev/null")
    opt.run(fmax=fmax, steps=max_steps)
    relaxed = ADAPTOR.get_structure(atoms)
    try:
        sg_after = SpacegroupAnalyzer(relaxed, symprec=1e-3).get_space_group_number()
    except Exception:
        sg_after = None
    E = float(atoms.get_potential_energy())
    fmax_final = float(np.linalg.norm(atoms.get_forces(), axis=1).max())
    return {
        "relaxed": relaxed, "energy_eV": E, "energy_per_atom": E/len(relaxed),
        "converged": bool(opt.converged()), "n_steps": int(opt.nsteps),
        "final_fmax": fmax_final, "reached_max_steps": bool(opt.nsteps >= max_steps),
        "sg_before": sg_before, "sg_after": sg_after,
        "symmetry_changed": (sg_before != sg_after),
    }


Smoke test

In [ ]:
from ase.build import bulk
from pymatgen.io.ase import AseAtomsAdaptor
s = AseAtomsAdaptor.get_structure(bulk("Cu"))
r = relax_atoms(s, PET_CALC, fmax=0.05, max_steps=5)
print("works:", r["energy_per_atom"])

CELL 6 — Relax the 38 parent seeds (consistent protocol, BEFORE substitution)

In [ ]:
# ============================================================================
# CELL 01 (BYPASS) — restore PET-relaxed seeds from zip + json
#   Replaces the 14-15 min seed relaxation. Cells 02/03 work unchanged.
# ============================================================================
import zipfile, os, glob, json

SEED_ZIP  = "relaxed_seeds.zip"          # your saved zip
SEED_JSON = "seed_relax_log.json"        # your saved log (optional but kept for provenance)

os.makedirs("relaxed_seeds", exist_ok=True)
with zipfile.ZipFile(SEED_ZIP) as z:
    z.extractall("relaxed_seeds")

# if the zip contained a nested folder, flatten it so cell:02's glob finds the files
nested = glob.glob("relaxed_seeds/**/*.vasp", recursive=True)
flat   = glob.glob("relaxed_seeds/*.vasp")
if nested and not flat:
    import shutil
    for f in nested:
        shutil.move(f, os.path.join("relaxed_seeds", os.path.basename(f)))

seed_files = glob.glob("relaxed_seeds/*.vasp")
print("relaxed seed structures restored:", len(seed_files))

# load the relaxation log (provenance; also lets you re-check convergence)
if os.path.exists(SEED_JSON):
    seed_log = json.load(open(SEED_JSON))
else:  # maybe it was inside the zip
    inzip = glob.glob("relaxed_seeds/**/seed_relax_log.json", recursive=True)
    seed_log = json.load(open(inzip[0])) if inzip else {}
print("seed log entries:", len(seed_log))

# sanity: every seed converged?
bad = [k for k, v in seed_log.items() if not v.get("converged", True)]
print("non-converged seeds:", bad if bad else "none")

assert seed_files, "No relaxed seed .vasp files found — check the zip path/contents"

CELL 7 — Map each seed to its (stoich, polymorph) so candidates find a template

In [ ]:
import csv, re
from math import gcd
from functools import reduce
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

F_SET={"La","Ce","Pr","Nd","Pm","Sm","Eu","Gd","Tb","Dy","Ho","Er","Tm","Yb","Lu"}
P_SET={"B","C","N","O","P","S","Se"}
CSYS = {"triclinic":"triclinic","monoclinic":"monoclinic","orthorhombic":"orthorhombic",
        "tetragonal":"tetragonal","trigonal":"trigonal","hexagonal":"hexagonal","cubic":"cubic"}

def sig_of(structure):
    el = structure.composition.get_el_amt_dict()
    counts={k:int(v) for k,v in el.items()}
    g=reduce(gcd, counts.values())
    red={k:v//g for k,v in counts.items()}
    fn=dn=pn=0
    for k,v in red.items():
        if k in F_SET: fn=v
        elif k in P_SET: pn=v
        else: dn=v
    return (fn,dn,pn)

# build template index: (stoich_sig, crystal_system) -> relaxed seed path
template_index = {}
for f in glob.glob("relaxed_seeds/*.vasp"):
    s = Structure.from_file(f)
    sig = sig_of(s)
    csys = SpacegroupAnalyzer(s, symprec=1e-2).get_crystal_system()
    template_index.setdefault((sig, csys), f)   # first seed for that (sig,system)

print("Template (stoich, crystal_system) keys available:")
for k in sorted(template_index): print("  ", k, "->", os.path.basename(template_index[k]))


CELL 8 — Substitute the 100 candidates into the matching relaxed seed templates








In [ ]:
import csv
os.makedirs("candidates_init", exist_ok=True)

def substitute(template_struct, f_el, d_el, p_el):
    """Replace the F/D/P species in the template by element role.
    Identifies which site element is the lanthanide / anion / TM and swaps."""
    s = template_struct.copy()
    # determine the template's role elements
    el_set = {str(e) for e in s.composition.elements}
    t_f = [e for e in el_set if e in F_SET]
    t_p = [e for e in el_set if e in P_SET]
    t_d = [e for e in el_set if e not in F_SET and e not in P_SET]
    mapping = {}
    if t_f: mapping[t_f[0]] = f_el
    if t_d: mapping[t_d[0]] = d_el
    if t_p: mapping[t_p[0]] = p_el
    s.replace_species(mapping)
    return s

rows = list(csv.DictReader(open(CSV_PATH)))
made, skipped = 0, []
candidate_meta = {}
for row in rows:
    sid = row["structure_id"]
    sig = (int(row["stoich_F"]), int(row["stoich_D"]), int(row["stoich_P"]))
    csys = row["polymorph"]   # plain crystal-system name
    key = (sig, csys)
    if key not in template_index:
        skipped.append((sid, key)); continue
    tmpl = Structure.from_file(template_index[key])
    try:
        cand = substitute(tmpl, row["F"], row["D"], row["P"])
        Poscar(cand).write_file(f"candidates_init/{sid}.vasp")
        candidate_meta[sid] = row
        made += 1
    except Exception as e:
        skipped.append((sid, str(e)))

print(f"Built {made} candidate starting structures; skipped {len(skipped)}")
if skipped[:5]: print("examples skipped:", skipped[:5])
json.dump(candidate_meta, open("candidate_meta.json","w"))


Cell 09: Relax all 100 candidates with PET-OAM-XL  (the main run)

In [ ]:

os.makedirs("candidates_relaxed", exist_ok=True)
FMAX_CAND = 0.005   # npj criterion; use 0.02 if you want the matbench-discovery setting

cand_log = []
init_files = sorted(glob.glob("candidates_init/*.vasp"))
print(f"Relaxing {len(init_files)} candidates with PET-OAM-XL ...")
for i, f in enumerate(init_files):
    sid = os.path.basename(f).replace(".vasp","")
    try:
        s = Structure.from_file(f)
        r = relax_atoms(s, PET_CALC, fmax=FMAX_CAND, max_steps=500, relax_cell=True)
        Poscar(r["relaxed"]).write_file(f"candidates_relaxed/{sid}.vasp")
        rec = {k:v for k,v in r.items() if k!="relaxed"}
        rec.update({"structure_id": sid, "status": "ok"})
        cand_log.append(rec)
    except Exception as e:
        cand_log.append({"structure_id": sid, "status": f"FAILED: {e}"})
    if (i+1) % 100 == 0:
        print(f"  {i+1}/{len(init_files)} done")
        json.dump(cand_log, open("pet_candidate_relax_log.json","w"),
                  indent=2, cls=NumpyEncoder)   # checkpoint progress

json.dump(cand_log, open("pet_candidate_relax_log.json","w"),
          indent=2, cls=NumpyEncoder)
ok = sum(1 for x in cand_log if x.get("status")=="ok")
print(f"\nDONE: {ok}/{len(init_files)} relaxed successfully")


In [ ]:
# ============================================================================
# CELL 9 — restore this batch's saved relaxation results
# ============================================================================
import json, zipfile, os, glob

os.makedirs("candidates_relaxed", exist_ok=True)
with zipfile.ZipFile(f"/content/pet_vasp_subset.zip") as z:
    z.extractall("candidates_relaxed")
print("relaxed structures:", len(glob.glob("candidates_relaxed/*.vasp")))

cand_log = json.load(open(f"/content/pet_json_subset.json"))
print("log records:", len(cand_log),
      "| ok:", sum(1 for x in cand_log if x.get("status")=="ok"))

In [ ]:
# ============================================================================
# CELL 1 — PET element references, hybrid scheme:
#   1) If a structure file exists in DFT_REF_DIR (your relaxed CONTCARs,
#      renamed <Symbol>.vasp), PET-relax YOUR structure.
#   2) Otherwise, PET-relax the ASE standard elemental ground state.
# All energies are PET -> internally consistent for the Paper-2 screening.
# ============================================================================
import os, glob, json
import pandas as pd
from pymatgen.core import Structure
from ase.build import bulk as ase_bulk
from pymatgen.io.ase import AseAtomsAdaptor

DFT_REF_DIR = "mp_reference_structures"   # contains Ag.vasp, C.vasp, ... N.vasp, B.vasp, Mn.vasp
MOLECULAR   = {"O", "N"}                   # molecule-in-a-box refs: never relax the cell

def elements_in_batch(csv_path):
    d = pd.read_csv(csv_path)
    els = set()
    for col in ["F", "D", "P"]:
        els |= set(d[col].astype(str))
    return sorted(els)

NEEDED = elements_in_batch("/content/merged_combinations_subset.csv")
print(f"Elements to reference: {len(NEEDED)}\n{NEEDED}\n")

# index your provided reference files by element symbol
ref_files = {}
for f in glob.glob(os.path.join(DFT_REF_DIR, "*.vasp")) + \
         glob.glob(os.path.join(DFT_REF_DIR, "*.cif")):
    ref_files[os.path.splitext(os.path.basename(f))[0]] = f
print(f"Your mp reference structures found: {sorted(ref_files)}\n")

ELEMENT_REF_ENERGY, ELEMENT_REF_LOG = {}, {}

for sym in NEEDED:
    try:
        if sym in ref_files:                                   # priority 1: YOUR structure
            struct = Structure.from_file(ref_files[sym])
            source = "mp_file"
        else:                                                  # priority 2: ASE standard
            struct = AseAtomsAdaptor.get_structure(ase_bulk(sym))
            source = "ase_bulk"
        relax_cell = sym not in MOLECULAR
        r = relax_atoms(struct, PET_CALC, fmax=0.005, max_steps=2000,
                        relax_cell=relax_cell)
        ELEMENT_REF_ENERGY[sym] = r["energy_per_atom"]
        log = {k: v for k, v in r.items() if k != "relaxed"}
        log["source"] = source
        ELEMENT_REF_LOG[sym] = log
        flag = "" if r["converged"] else "  <-- not converged"
        print(f"{sym:3} [{source:8}] E/atom={r['energy_per_atom']:.5f} "
              f"conv={r['converged']} symchg={r['symmetry_changed']}{flag}")
    except Exception as e:
        print(f"{sym:3} FAILED: {e}")

json.dump(ELEMENT_REF_ENERGY, open("/content/pet_element_refs.json", "w"), indent=2)
json.dump(ELEMENT_REF_LOG, open("pet_element_refs_ALL.json", "w"),
          indent=2, cls=NumpyEncoder)

still_missing = sorted(set(NEEDED) - set(ELEMENT_REF_ENERGY))
print(f"\nBuilt {len(ELEMENT_REF_ENERGY)}/{len(NEEDED)} references "
      f"({sum(1 for l in ELEMENT_REF_LOG.values() if l['source']=='dft_file')} from your files, "
      f"{sum(1 for l in ELEMENT_REF_LOG.values() if l['source']=='ase_bulk')} from ASE standards)")
print("STILL MISSING:", still_missing if still_missing else "none")

Cell : 10 (formation energy)

In [ ]:
# ============================================================================
# CELL 1 (BYPASS) — load PET element references from saved JSONs
#   Replaces the 14-15 min reference relaxation. No dft_reference_structures
#   folder needed. Cell 2 works unchanged.
# ============================================================================
import json
import pandas as pd

REF_ENERGY_JSON = "pet_element_ref_energies.json"
REF_LOG_JSON    = "pet_element_refs_ALL.json"

ELEMENT_REF_ENERGY = json.load(open(REF_ENERGY_JSON))
ELEMENT_REF_LOG    = json.load(open(REF_LOG_JSON))
print(f"Loaded {len(ELEMENT_REF_ENERGY)} element reference energies")

# ---- coverage check for THIS batch (the alignment guarantee) ----
def elements_in_batch(csv_path):
    d = pd.read_csv(csv_path)
    els = set()
    for col in ["F", "D", "P"]:
        els |= set(d[col].astype(str))
    return sorted(els)

NEEDED  = elements_in_batch(f"batch_1.csv")
missing = sorted(set(NEEDED) - set(ELEMENT_REF_ENERGY))
print(f"Batch 1 needs {len(NEEDED)} elements | missing: {missing if missing else 'none'}")

# ---- quality guards (same info the live run printed) ----
nonconv = sorted(s for s, l in ELEMENT_REF_LOG.items() if not l.get("converged", True))
print("non-converged refs:", nonconv if nonconv else "none")
print("sources:", {src: sum(1 for l in ELEMENT_REF_LOG.values() if l.get('source')==src)
                   for src in {"dft_file", "ase_bulk"}})

# ---- top-up: relax ONLY elements this batch needs that aren't in the JSONs ----
if missing:
    from ase.build import bulk as ase_bulk
    from pymatgen.io.ase import AseAtomsAdaptor
    MOLECULAR = {"O", "N"}
    for sym in missing:
        try:
            struct = AseAtomsAdaptor.get_structure(ase_bulk(sym))
            r = relax_atoms(struct, PET_CALC, fmax=0.005, max_steps=2000,
                            relax_cell=(sym not in MOLECULAR))
            ELEMENT_REF_ENERGY[sym] = r["energy_per_atom"]
            log = {k: v for k, v in r.items() if k != "relaxed"}
            log["source"] = "ase_bulk"
            ELEMENT_REF_LOG[sym] = log
            print(f"[top-up] {sym}: E/atom={r['energy_per_atom']:.5f} conv={r['converged']}")
        except Exception as e:
            print(f"[top-up] {sym} FAILED: {e}")
    # persist the updated reference set for future batches
    json.dump(ELEMENT_REF_ENERGY, open(REF_ENERGY_JSON, "w"), indent=2)
    json.dump(ELEMENT_REF_LOG, open(REF_LOG_JSON, "w"), indent=2, cls=NumpyEncoder)
    print("Reference JSONs updated with top-ups.")

In [ ]:
# ============================================================================
# CELL 1.5 — RESCUE: re-relax only genuinely unconverged candidates
#             (starts from the already-relaxed geometry -> cheap continuation)
# ============================================================================
from pymatgen.core import Structure
from pymatgen.io.vasp import Poscar

FMAX = 0.005

def tier_of(rec):
    f = float(rec.get("final_fmax", 9e9))
    if f <= FMAX:     return "converged_strict"
    if f <= 2*FMAX:   return "near_converged"
    return "unconverged"

to_rescue = [rec for rec in cand_log
             if rec.get("status") == "ok" and tier_of(rec) == "unconverged"]
print(f"Rescue candidates: {len(to_rescue)}")
for rec in to_rescue:
    print(f"  {rec['structure_id']:32} fmax={float(rec['final_fmax']):.5f}")

for rec in to_rescue:
    sid = rec["structure_id"]
    fpath = f"candidates_relaxed/{sid}.vasp"
    try:
        s = Structure.from_file(fpath)             # CONTINUE from current geometry
        r = relax_atoms(s, PET_CALC, fmax=FMAX, max_steps=2000, relax_cell=True)
        Poscar(r["relaxed"]).write_file(fpath)     # overwrite with improved geometry
        # update the log record in place
        for k in ("energy_eV","energy_per_atom","converged","n_steps",
                  "final_fmax","reached_max_steps","sg_before","sg_after",
                  "symmetry_changed"):
            rec[k] = r[k] if k in r else rec.get(k)
        rec["rescued"] = True
        print(f"{sid:32} -> fmax={r['final_fmax']:.5f} conv={r['converged']}")
    except Exception as e:
        rec["rescued"] = f"FAILED: {e}"
        print(f"{sid:32} rescue FAILED: {e}")

# save the updated log (overwrites the batch log with rescued values)
json.dump(cand_log, open(f"/content/pet_json_subset.json","w"),
          indent=2, cls=NumpyEncoder)
print("Rescue pass complete; log updated.")

In [ ]:
# ============================================================================
# CELL 2 — Formation energies for batch {BATCH}, honest convergence tiers
# ============================================================================
import json, os, csv
from pymatgen.core import Structure

FMAX = 0.005
REF = ELEMENT_REF_ENERGY

NONCONVERGED_REFS = {s for s,l in ELEMENT_REF_LOG.items() if not l.get("converged", True)}
ASEBULK_REFS      = {s for s,l in ELEMENT_REF_LOG.items() if l.get("source")=="ase_bulk"}
if NONCONVERGED_REFS:
    print("WARNING: non-converged refs:", sorted(NONCONVERGED_REFS))

def classify(rec):
    f = float(rec.get("final_fmax", 9e9))
    if f <= FMAX:     return "converged_strict"
    if f <= 2*FMAX:   return "near_converged"
    return "unconverged"

def e_form_per_atom(structure, total_E):
    comp = structure.composition
    ref_sum, flags = 0.0, []
    for el, amt in comp.get_el_amt_dict().items():
        el = str(el)
        e = REF.get(el)
        if e is None:
            return None, ["missing:"+el]
        if el in NONCONVERGED_REFS: flags.append("noncvg_ref:"+el)
        if el in ASEBULK_REFS:      flags.append("asebulk_ref:"+el)
        ref_sum += amt * e
    return (total_E - ref_sum)/comp.num_atoms, flags

counts = {"converged_strict":0,"near_converged":0,"unconverged":0}
done, skipped = 0, 0
for rec in cand_log:
    if rec.get("status") != "ok":
        skipped += 1; continue
    sid = rec["structure_id"]
    fpath = f"candidates_relaxed/{sid}.vasp"
    if not os.path.exists(fpath):
        rec["e_form_per_atom"] = None; skipped += 1; continue
    tier = classify(rec); counts[tier] += 1
    rec["convergence_tier"] = tier
    s = Structure.from_file(fpath)
    ef, flags = e_form_per_atom(s, rec["energy_eV"])
    rec["e_form_per_atom"] = ef
    rec["e_form_flags"] = flags
    if ef is None: skipped += 1
    else: done += 1

json.dump(cand_log, open(f"pet_candidate_formation_subset.json","w"),
          indent=2, cls=NumpyEncoder)

with open(f"pet_subset_formation_energies.csv","w",newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["structure_id","energy_eV","e_form_per_atom",
                "convergence_tier","final_fmax","flags"])
    for rec in cand_log:
        if rec.get("status")=="ok":
            w.writerow([rec["structure_id"], rec.get("energy_eV"),
                        rec.get("e_form_per_atom"), rec.get("convergence_tier"),
                        rec.get("final_fmax"), ";".join(rec.get("e_form_flags",[]))])

print("Tiers:", counts)
print(f"E_form: {done} computed, {skipped} skipped")
print(f"Wrote pet_subset_formation_energies.csv")

Force

In [ ]:
# ============================================================================
# CELL 4 — Off-equilibrium force benchmark, PET-OAM-XL
# Input : candidates_relaxed/*.vasp  +  batch_{BATCH}_formation_energies.csv
# Output: batch_{BATCH}_formation_forces.csv  (formation energy + force columns)
# ============================================================================
import os, json, zlib, warnings
import numpy as np
import pandas as pd
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor

warnings.filterwarnings("ignore")

BATCH        = 1
RELAXED_DIR  = "candidates_relaxed"
IN_CSV       = f"pet_subset_formation_energies.csv"
OUT_CSV      = f"pet_subset_formation_forces.csv"
OUT_JSON     = f"pet_force_benchmark_subset.json"
META_JSON    = f"pet_force_meta_subset.json"

RATTLE_SIGMA = 0.03          # Angstrom, per-Cartesian-component Gaussian
N_RATTLE     = 3
EV_A3_TO_GPA = 160.21766208
_ADP = AseAtomsAdaptor()

assert "PET_CALC" in dir(), "run the calculator cell first"
assert os.path.exists(IN_CSV), f"{IN_CSV} not found — run the formation-energy cell"


def _single_point(atoms):
    """Energy, forces, hydrostatic pressure at a fixed geometry."""
    atoms.calc = PET_CALC
    E = float(atoms.get_potential_energy())
    F = np.asarray(atoms.get_forces(), dtype=np.float64)
    try:
        S = np.asarray(atoms.get_stress(voigt=True), dtype=np.float64)
        P = float(-np.mean(S[:3]) * EV_A3_TO_GPA)
    except Exception:
        P = None
    return E, F, P


def _stats(F):
    mag = np.linalg.norm(F, axis=1)
    return (float(np.sqrt(np.mean(np.sum(F ** 2, axis=1)))),   # rms over atoms
            float(mag.max()), float(mag.mean()),
            float(np.linalg.norm(F.sum(axis=0))))              # net (drift)


def force_benchmark(struct, sid):
    n = len(struct)
    base = _ADP.get_atoms(struct)

    E_eq, F_eq, P_eq = _single_point(base.copy())
    rms_eq, max_eq, _, net_eq = _stats(F_eq)

    # crc32 of the id -> identical rattles across sessions and across models
    seed0 = zlib.crc32(sid.encode("utf-8")) & 0xFFFFFFFF
    rms_r, max_r, mean_r, dE_r = [], [], [], []
    for k in range(N_RATTLE):
        rng = np.random.default_rng(seed0 + k)
        a = base.copy()
        a.set_positions(a.get_positions() + rng.normal(0.0, RATTLE_SIGMA, (n, 3)))
        E_k, F_k, _ = _single_point(a)
        r, m, mu, _ = _stats(F_k)
        rms_r.append(r); max_r.append(m); mean_r.append(mu)
        dE_r.append((E_k - E_eq) / n * 1000.0)

    return {
        "structure_id": sid, "n_atoms": n,
        "energy_sp_eV": E_eq,
        "f_rms_eq": rms_eq, "f_max_eq": max_eq, "f_net_eq": net_eq,
        "pressure_eq_GPa": P_eq,
        "f_rms_rattled":  float(np.mean(rms_r)),
        "f_max_rattled":  float(np.max(max_r)),
        "f_mean_rattled": float(np.mean(mean_r)),
        "f_rms_rattled_spread": float(np.std(rms_r, ddof=1)) if N_RATTLE > 1 else 0.0,
        "dE_rattle_meV_per_atom": float(np.mean(dE_r)),
        "force_status": "ok",
    }


df = pd.read_csv(IN_CSV)
print(f"{len(df)} rows in {IN_CSV}")

results = json.load(open(OUT_JSON)) if os.path.exists(OUT_JSON) else []   # resume
done = {r["structure_id"] for r in results}
if done:
    print(f"resuming — {len(done)} already benchmarked")

todo = [s for s in df["structure_id"].astype(str) if s not in done]
for i, sid in enumerate(todo, 1):
    fp = os.path.join(RELAXED_DIR, f"{sid}.vasp")
    if not os.path.exists(fp):
        results.append({"structure_id": sid, "force_status": "missing_structure"})
        continue
    try:
        results.append(force_benchmark(Structure.from_file(fp), sid))
    except Exception as e:
        results.append({"structure_id": sid,
                        "force_status": f"FAILED: {type(e).__name__}: {e}"})
        print(f"  {sid}: {type(e).__name__}: {e}")
    if i % 20 == 0 or i == len(todo):
        json.dump(results, open(OUT_JSON, "w"), indent=2)
        print(f"  {i}/{len(todo)}")

json.dump(results, open(OUT_JSON, "w"), indent=2)

fdf = pd.DataFrame(results).drop_duplicates("structure_id", keep="last")
out = df.merge(fdf, on="structure_id", how="left")

# consistency check: single-point energy vs the energy the optimiser reported
if "energy_eV" in out.columns:
    out["dE_sp_vs_relax_meV_per_atom"] = (
        (out["energy_sp_eV"] - out["energy_eV"]) / out["n_atoms"] * 1000.0)

out.to_csv(OUT_CSV, index=False)
json.dump({"model": "PET-OAM-XL", "force_base": "self_relaxed",
           "rattle_sigma_A": RATTLE_SIGMA, "n_rattle": N_RATTLE,
           "rattle_seed": "zlib.crc32(structure_id)",
           "device": str(globals().get("DEVICE", "unknown"))},
          open(META_JSON, "w"), indent=2)

ok = (out["force_status"] == "ok").sum()
print(f"\n{ok}/{len(out)} benchmarked -> {OUT_CSV}")
print(out.loc[out.force_status == "ok",
              ["f_rms_eq", "f_net_eq", "f_rms_rattled",
               "dE_rattle_meV_per_atom", "pressure_eq_GPa"]].describe().T)

Phonon


In [ ]:
# ============================================================================
# CELL 11 — PHONON THERMODYNAMICS
#
# Protocol from the umlip / phonondb MLIP benchmark (calc_phonopy.py):
#   displacements 0.01 A with is_diagonal=False; drift removed; force
#   constants symmetrised; dynamical stability judged at COMMENSURATE
#   q-points, which the supercell resolves exactly, not at interpolated
#   mesh points.
#
# Symmetry is refined once (refine_symmetry) to snap the numerical wobble in
# the MLIP-relaxed cell, then the relaxation runs UNCONSTRAINED. Verified on
# LaCoO3: unconstrained reaches 5.6e-5 eV/A and keeps sg=2, while ASE
# FixSymmetry + FrechetCellFilter stalls at 1.6e-2 at the same energy to
# within 0.00 meV/atom. The constraint blocks convergence without changing
# the answer, so it is not used.
#
# Model-agnostic: set CALC and MODEL_TAG. Nothing below is PET-specific.
# ============================================================================
import os, json, gc, math, time, traceback, warnings
import numpy as np
import pandas as pd
from ase import Atoms
from ase.optimize import FIRE
from ase.filters import FrechetCellFilter
from ase.spacegroup.symmetrize import refine_symmetry, check_symmetry
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.harmonic.dynmat_to_fc import get_commensurate_points

try:
    import torch
except ImportError:
    torch = None
warnings.filterwarnings("ignore")

CALC      = PET_CALC                 # <-- swap for any ASE calculator
MODEL_TAG = "pet-oam-xl"

RELAXED_DIR = "candidates_relaxed"
IN_CSV      = ("/content/pet_subset_formation_forces.csv"
               if os.path.exists("/content/pet_subset_formation_forces.csv")
               else "/content/pet_subset_formation_energies.csv")
YAML_DIR    = f"phonopy_{MODEL_TAG}"
OUT_JSON    = f"{MODEL_TAG}_phonon_55.json"
OUT_CSV     = f"{MODEL_TAG}_phonon_55.csv"

SYMPREC      = 1e-2     # symmetry refined once at this tolerance
REFINE_DMAX  = 0.10     # A; reject the refinement if it moves an atom further
DISP         = 0.01     # A, umlip benchmark value
PH_FMAX      = 1e-4     # eV/A relaxation target
PH_GATE      = 5e-4     # eV/A; above this no phonons are reported
PH_STEPS     = 2000
LMIN         = 10.0     # A, minimum supercell edge
MIN_REP      = 2        # a 1-thick supercell lets an atom see its own image
MAX_SC_ATOMS = 500      # skip rather than thin the supercell
MAX_DISP     = 600
MESH         = 45.0
TEMPS        = (0.0, 75.0, 150.0, 300.0, 600.0)
IMAG_TOL     = -0.05    # THz
EXCLUDE_TIERS = {"unconverged"}

os.makedirs(YAML_DIR, exist_ok=True)
_ADP = AseAtomsAdaptor()


def free_mem():
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


def mesh_from_length(lattice, length=MESH):
    rec = np.linalg.inv(np.asarray(lattice)).T
    return np.maximum(np.rint(np.linalg.norm(rec, axis=1) * length).astype(int), 1).tolist()


def supercell_matrix(abc, natoms):
    m = [max(int(math.ceil(LMIN / x)), MIN_REP) for x in abc]
    while natoms * int(np.prod(m)) > MAX_SC_ATOMS and max(m) > MIN_REP:
        m[int(np.argmax(m))] -= 1
    return m


def forces_on(cell):
    """One displaced supercell. Retries once after clearing GPU memory."""
    for attempt in (0, 1):
        try:
            at = Atoms(cell=cell.cell, symbols=cell.symbols,
                       scaled_positions=cell.scaled_positions, pbc=True)
            at.calc = CALC
            F = np.asarray(at.get_forces(), dtype=np.float64)
            del at
            return F
        except RuntimeError:
            free_mem()
            if attempt:
                raise


def phonon_one(sid, verbose=True):
    t0 = time.time()
    rec = {"structure_id": sid}

    atoms = _ADP.get_atoms(Structure.from_file(os.path.join(RELAXED_DIR, f"{sid}.vasp")))
    rec["sg_input"] = check_symmetry(atoms, symprec=SYMPREC).number

    pos0 = atoms.get_positions().copy()
    refine_symmetry(atoms, symprec=SYMPREC)
    rec["refine_max_disp_A"] = float(np.abs(atoms.get_positions() - pos0).max())
    if rec["refine_max_disp_A"] > REFINE_DMAX:
        atoms = _ADP.get_atoms(Structure.from_file(os.path.join(RELAXED_DIR, f"{sid}.vasp")))
        rec["refine_rejected"] = True
    rec["sg"] = check_symmetry(atoms, symprec=SYMPREC).number

    atoms.calc = CALC
    FIRE(FrechetCellFilter(atoms), logfile=None).run(fmax=PH_FMAX, steps=PH_STEPS)
    rec["prerelax_fmax"] = float(np.linalg.norm(atoms.get_forces(), axis=1).max())
    rec["prerelax_E_per_atom"] = float(atoms.get_potential_energy()) / len(atoms)
    rec["sg_after_relax"] = check_symmetry(atoms, symprec=SYMPREC).number
    if rec["prerelax_fmax"] > PH_GATE:
        rec["status"] = f"rejected: prerelax fmax {rec['prerelax_fmax']:.2e} > {PH_GATE:.0e}"
        return rec

    scm = supercell_matrix(atoms.cell.cellpar()[:3], len(atoms))
    n_sc = len(atoms) * int(np.prod(scm))
    rec.update(n_atoms_cell=len(atoms), supercell="x".join(map(str, scm)),
               n_atoms_supercell=n_sc, min_repetition=min(scm))
    if n_sc > MAX_SC_ATOMS:
        rec["status"] = f"skipped: supercell needs {n_sc} atoms (> {MAX_SC_ATOMS})"
        return rec

    ph = Phonopy(PhonopyAtoms(cell=atoms.get_cell(),
                              scaled_positions=atoms.get_scaled_positions(),
                              symbols=atoms.get_chemical_symbols()),
                 supercell_matrix=np.diag(scm), primitive_matrix="auto",
                 symprec=SYMPREC)
    ph.generate_displacements(distance=DISP, is_diagonal=False)
    cells = ph.supercells_with_displacements
    rec["n_displacements"] = len(cells)
    if verbose:
        print(f"      cell {len(atoms)} atoms, sg {rec['sg']} -> supercell "
              f"{'x'.join(map(str,scm))} = {n_sc} atoms, {len(cells)} displacements")
    if len(cells) > MAX_DISP:
        rec["status"] = f"skipped: {len(cells)} displacements (> {MAX_DISP})"
        return rec

    forces, drift = [], []
    for i, sc in enumerate(cells):
        F = forces_on(sc)
        drift.append(float(np.linalg.norm(F.sum(axis=0))))
        forces.append(F - F.mean(axis=0))
        if (i + 1) % 5 == 0:
            free_mem()
    free_mem()
    rec["force_drift_max"] = float(np.max(drift))

    ph.forces = forces
    ph.produce_force_constants()
    ph.symmetrize_force_constants()
    ph.save(os.path.join(YAML_DIR, f"{sid}.yaml"), settings={"force_constants": True})

    fq = np.array([ph.get_frequencies(q)
                   for q in get_commensurate_points(ph.supercell_matrix)])
    rec.update(freq_min_commensurate_THz=float(fq.min()),
               freq_max_THz=float(fq.max()),
               imag_fraction=float(np.mean(fq < IMAG_TOL)),
               dynamically_stable=bool(fq.min() >= IMAG_TOL))

    ph.run_mesh(mesh_from_length(ph.primitive.cell), is_gamma_center=True)
    ph.run_thermal_properties(temperatures=TEMPS)
    tp = ph.get_thermal_properties_dict()
    npc = len(ph.primitive.symbols)
    for T, F, S, C in zip(TEMPS, tp["free_energy"], tp["entropy"], tp["heat_capacity"]):
        k = int(T)
        rec[f"F_vib_{k}K_meV_per_atom"] = float(F / 96.48533212331 / npc * 1000)
        rec[f"S_{k}K_J_K_mol_atom"]     = float(S / npc)
        rec[f"Cv_{k}K_J_K_mol_atom"]    = float(C / npc)
    rec["status"] = "ok"
    rec["walltime_s"] = round(time.time() - t0, 1)
    return rec


df = pd.read_csv(IN_CSV)
df["structure_id"] = df["structure_id"].astype(str)
sel = df[~df.get("convergence_tier", pd.Series("", index=df.index))
         .astype(str).isin(EXCLUDE_TIERS)]
sel = sel[pd.to_numeric(sel["e_form_per_atom"], errors="coerce").notna()]

results = json.load(open(OUT_JSON)) if os.path.exists(OUT_JSON) else []
done = {r["structure_id"] for r in results}
todo = [s for s in sel["structure_id"] if s not in done
        and os.path.exists(os.path.join(RELAXED_DIR, f"{s}.vasp"))]
print(f"{len(df)} rows | {len(sel)} eligible | {len(done)} done | {len(todo)} to run")

t0 = time.time()
for i, sid in enumerate(todo, 1):
    free_mem()
    print(f"[{i}/{len(todo)}] {sid}")
    try:
        r = phonon_one(sid)
    except Exception as e:
        r = {"structure_id": sid, "status": f"FAILED: {type(e).__name__}: {e}",
             "traceback": traceback.format_exc()}
        print(traceback.format_exc())            # full text, never truncated
        free_mem()
    results.append(r)
    json.dump(results, open(OUT_JSON, "w"), indent=2)
    el = time.time() - t0
    print(f"      {str(r.get('status'))[:70]}  "
          f"fmin={r.get('freq_min_commensurate_THz')}  "
          f"| {el/60:.1f}m, ETA {el/i*(len(todo)-i)/60:.1f}m")

pdf = pd.DataFrame(results).drop_duplicates("structure_id", keep="last")
df.merge(pdf, on="structure_id", how="left").to_csv(OUT_CSV, index=False)
g = pdf[pdf.get("status", pd.Series(dtype=str)) == "ok"]
print(f"\n{len(g)}/{len(pdf)} ok -> {OUT_CSV}")
if len(g):
    print(f"dynamically stable: {int(g.dynamically_stable.sum())}/{len(g)}")
    print(f"min supercell repetition: {int(g.min_repetition.min())} (must be >= {MIN_REP})")
    print(f"max force drift: {g.force_drift_max.max():.2e} eV/A")
    print(f"max prerelax fmax: {g.prerelax_fmax.max():.2e} eV/A (gate {PH_GATE:.0e})")
    print(f"space groups: {dict(sorted(g.sg.value_counts().items()))}")
if len(g) < len(pdf):
    cols = [c for c in ["structure_id", "sg", "refine_max_disp_A", "prerelax_fmax",
                        "supercell", "n_atoms_supercell", "n_displacements", "status"]
            if c in pdf.columns]
    print("\nnot ok:")
    print(pdf[pdf.get("status", pd.Series(dtype=str)) != "ok"][cols].to_string(index=False))

In [ ]:
# ============================================================================
# CELL 11b — RERUN the 6 compounds that failed on memory (GPU only)
#   3 were OOM in a session that had accumulated memory; a clean restart with
#   expandable_segments should let them through.
#   3 were refused by the 500-atom cap; the cap is raised so they attempt.
#   No CPU fallback: anything that still OOMs is recorded as excluded.
#   Requires CELL 11 to have been run in this session.
# ============================================================================
import json, time
import numpy as np
import pandas as pd
from ase import Atoms

RERUN = ["HoFe5P3__orthorhombic", "Nd2Cu2O5__orthorhombic", "LaCo3B2__orthorhombic",
         "Nd2Mo2O7__cubic", "La2Zr2O7__cubic", "Lu3Fe5O12__cubic"]

MAX_SC_ATOMS = 1400          # raised so all six attempt rather than skip


def forces_on(cell):                     # overrides the CELL 11 version
    """GPU only. On OOM, free memory and raise with the supercell size named."""
    at = Atoms(cell=cell.cell, symbols=cell.symbols,
               scaled_positions=cell.scaled_positions, pbc=True)
    at.calc = CALC
    try:
        return np.asarray(at.get_forces(), dtype=np.float64)
    except RuntimeError as e:
        free_mem()
        if "out of memory" in str(e).lower():
            raise RuntimeError(f"GPU out of memory at {len(at)} atoms") from None
        raise


results = json.load(open(OUT_JSON))
results = [r for r in results if r["structure_id"] not in set(RERUN)]
print(f"kept {len(results)} records | recomputing {len(RERUN)}\n")

t0 = time.time()
for i, sid in enumerate(RERUN, 1):
    free_mem()
    print(f"[{i}/{len(RERUN)}] {sid}")
    try:
        r = phonon_one(sid)
    except Exception as e:
        r = {"structure_id": sid, "status": f"FAILED: {type(e).__name__}: {e}"}
        free_mem()
    results.append(r)
    json.dump(results, open(OUT_JSON, "w"), indent=2)
    print(f"      {str(r.get('status'))[:80]}  "
          f"fmin={r.get('freq_min_commensurate_THz')}  "
          f"| {(time.time()-t0)/60:.1f}m elapsed")

df = pd.read_csv(IN_CSV)
df["structure_id"] = df["structure_id"].astype(str)
pdf = pd.DataFrame(results).drop_duplicates("structure_id", keep="last")
df.merge(pdf, on="structure_id", how="left").to_csv(OUT_CSV, index=False)

g = pdf[pdf.get("status", pd.Series(dtype=str)) == "ok"]
print(f"\n{len(g)}/{len(pdf)} ok -> {OUT_CSV}, {OUT_JSON}")
if len(g):
    print(f"dynamically stable: {int(g.dynamically_stable.sum())}/{len(g)}")
    print(f"min supercell repetition: {int(g.min_repetition.min())} (must be >= 2)")
    print(f"max force drift: {g.force_drift_max.max():.2e} eV/A")
if len(g) < len(pdf):
    cols = [c for c in ["structure_id", "sg", "supercell", "n_atoms_supercell",
                        "n_displacements", "status"] if c in pdf.columns]
    print("\nexcluded (report these in the SI with their required supercell):")
    print(pdf[pdf.get("status", pd.Series(dtype=str)) != "ok"][cols]
          .assign(status=lambda x: x.status.str.slice(0, 60)).to_string(index=False))

phonon imaginery figure

In [ ]:
# ============================================================================
# CELL 12 — PHONON BAND FIGURES (12 compounds absent from Materials Project)
#
# Reloads the force constants saved by CELL 11 — no recomputation. Band paths
# from SeeK-path. Short path segments are given a minimum on-screen width and
# crowded tick labels staggered, so labels never overprint. Titles carry the
# COMPUTED space group, not the seed polymorph tag.
# ============================================================================
!pip install -q seekpath

import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import phonopy
from phonopy.harmonic.dynmat_to_fc import get_commensurate_points
warnings.filterwarnings("ignore")
_TRAPZ = getattr(np, "trapezoid", None) or np.trapz

MODEL_TAG = "pet-oam-xl"
YAML_DIR  = f"phonopy_{MODEL_TAG}"
OUT_DIR   = "figs_phonon"

COMPOUNDS = [
    "Eu2FeO4__tetragonal",  "CeNiO4__orthorhombic", "YbNiO2__monoclinic",
    "YbNiB__tetragonal",    "EuNi3C2__hexagonal",   "LaCo3B2__orthorhombic",
    "La2Zr2Se5__orthorhombic", "Tm2PtN3__orthorhombic", "SmFeS2__orthorhombic",
    "Sm2Co2Se3__monoclinic", "Dy2VO4__tetragonal",  "HoMoB3__orthorhombic",
]

NPOINTS, MESH_LEN = 201, 45.0
MIN_SEG_FRAC, CROWD_FRAC = 0.06, 0.085
PDOS_MAX_ATOMS = 40

matplotlib.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans", "Arial"],
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 9,
    "legend.fontsize": 6.5, "axes.linewidth": 0.7,
    "xtick.direction": "in", "ytick.direction": "in",
    "pdf.fonttype": 42, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})
os.makedirs(OUT_DIR, exist_ok=True)


def band_data(ph):
    """SeeK-path bands. Labels folded at path breaks ('A|B'); every segment
    given a minimum width so short segments keep their labels legible."""
    ph.auto_band_structure(npoints=NPOINTS, with_eigenvectors=False)
    bs = ph.band_structure
    dist = [np.asarray(x) for x in bs.distances]
    freq = [np.asarray(x) for x in bs.frequencies]
    L, pc, n = list(bs.labels), list(bs.path_connections), len(dist)
    labels, li = [L[0]], 0
    for i in range(n):
        if i < n - 1 and not pc[i]:
            labels.append(f"{L[li+1]}|{L[li+2]}"); li += 2
        else:
            labels.append(L[li+1]); li += 1
    seg = np.array([d[-1] - d[0] for d in dist], float)
    tot = seg.sum() or 1.0
    tgt = np.maximum(seg, MIN_SEG_FRAC * tot); tgt *= tot / tgt.sum()
    edges = np.concatenate([[0.0], np.cumsum(tgt)])
    xs = [edges[i] + ((np.asarray(d) - d[0]) / seg[i] if seg[i] > 0
                      else np.zeros_like(d, float)) * tgt[i]
          for i, d in enumerate(dist)]
    return xs, freq, list(edges), labels


def dos_data(ph):
    rec = np.linalg.inv(np.asarray(ph.primitive.cell)).T
    mesh = np.maximum(np.rint(np.linalg.norm(rec, axis=1) * MESH_LEN).astype(int), 1).tolist()
    npc = len(ph.primitive.symbols)
    want = npc <= PDOS_MAX_ATOMS
    ph.run_mesh(mesh, is_gamma_center=True, with_eigenvectors=want,
                is_mesh_symmetry=not want)
    ph.run_total_dos(freq_pitch=0.05)
    fp, td = np.asarray(ph.total_dos.frequency_points), np.asarray(ph.total_dos.dos)
    pfp = pdos = None; syms = []
    if want:
        ph.run_projected_dos(freq_pitch=0.05)
        raw = np.asarray(ph.projected_dos.projected_dos)
        pfp = np.asarray(ph.projected_dos.frequency_points)
        el = [str(s) for s in ph.primitive.symbols]
        syms = sorted(set(el), key=el.index)
        pdos = np.array([raw[[i for i, e in enumerate(el) if e == s]].sum(axis=0)
                         for s in syms])
    return fp, td, pfp, pdos, syms, mesh, float(_TRAPZ(td, fp)), 3 * npc


def draw(ax, axd, D, ylabel=True, legend=True):
    xs, freq, edges, labels = D["bands"]
    for sx, sf in zip(xs, freq):
        ax.plot(sx, sf, color="0.15", lw=0.9, zorder=3)
    for x in edges[1:-1]:
        ax.axvline(x, color="0.65", lw=0.5, zorder=1)
    ax.axhline(0, color="0.4", lw=0.6, ls=":", zorder=1)
    ax.set_xlim(edges[0], edges[-1])
    ax.set_xticks(edges); ax.set_xticklabels([""] * len(edges))
    span, row, prev = edges[-1] - edges[0], 0, None
    for x, l in zip(edges, labels):
        row = 1 - row if (prev is not None and (x - prev) / span < CROWD_FRAC) else 0
        ax.text(x, -0.030 - 0.085 * row, l, transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=8)
        prev = x
    if ylabel:
        ax.set_ylabel("Frequency (THz)")
    allf = np.concatenate([f.ravel() for f in freq])
    pad = 0.06 * (allf.max() - min(allf.min(), 0))
    ax.set_ylim(min(allf.min(), 0) - pad, allf.max() + pad)

    fp, td, pfp, pdos, syms = D["dos"][:5]
    axd.fill_betweenx(fp, 0, td, color="0.85", lw=0, zorder=1)
    axd.plot(td, fp, color="0.15", lw=0.8, zorder=3)
    if pdos is not None:
        for i, s in enumerate(syms):
            axd.plot(pdos[i], pfp, lw=0.7, zorder=2, label=s)
        if legend:
            axd.legend(frameon=False, loc="upper right", handlelength=1.0,
                       borderpad=0.1, labelspacing=0.2)
    axd.axhline(0, color="0.4", lw=0.6, ls=":")
    axd.set_xlim(0, None); axd.set_xticks([])
    axd.tick_params(labelleft=False); axd.set_xlabel("PhDOS")


data, rows = {}, []
for sid in COMPOUNDS:
    fp = os.path.join(YAML_DIR, f"{sid}.yaml")
    if not os.path.exists(fp):
        print(f"{sid:28} no force constants — run CELL 11 first"); continue
    ph = phonopy.load(fp, symprec=SYMPREC)
    sym = ph.symmetry.dataset
    sg = sym.number if hasattr(sym, "number") else sym["number"]
    sg_sym = sym.international if hasattr(sym, "international") else sym["international"]
    D = {"bands": band_data(ph), "dos": dos_data(ph), "sg": sg, "sg_symbol": sg_sym}
    fq = np.array([ph.get_frequencies(q)
                   for q in get_commensurate_points(ph.supercell_matrix)])
    data[sid] = D

    formula = sid.split("__")[0]
    fig = plt.figure(figsize=(3.5, 2.7))
    gs = fig.add_gridspec(1, 2, width_ratios=[3.1, 1.0], wspace=0.06)
    ax = fig.add_subplot(gs[0]); axd = fig.add_subplot(gs[1], sharey=ax)
    draw(ax, axd, D)
    ax.set_title(f"{formula}   {sg_sym} (#{sg})")
    fig.savefig(os.path.join(OUT_DIR, f"{sid}_bands.pdf"))
    fig.savefig(os.path.join(OUT_DIR, f"{sid}_bands.png"), dpi=600)
    plt.close(fig)

    rows.append({"structure_id": sid, "sg_symbol": sg_sym, "sg": sg,
                 "supercell": "x".join(map(str, np.diag(ph.supercell_matrix))),
                 "n_atoms_primitive": len(ph.primitive.symbols),
                 "freq_min_commensurate_THz": float(fq.min()),
                 "acoustic_residual_THz": float(np.abs(np.sort(D["bands"][1][0][0])[:3]).max()),
                 "dos_mesh": "x".join(map(str, D["dos"][5])),
                 "dos_integral": D["dos"][6], "dos_expected_3N": D["dos"][7]})
    print(f"{sid:28} {sg_sym:11} fmin={fq.min():+.4f} THz -> {OUT_DIR}/{sid}_bands.pdf")

if data:
    ids = [s for s in COMPOUNDS if s in data]
    ncol, nrow = 3, int(np.ceil(len(ids) / 3))
    fig = plt.figure(figsize=(7.2, 2.25 * nrow))
    outer = fig.add_gridspec(nrow, ncol, hspace=0.70, wspace=0.30)
    for i, sid in enumerate(ids):
        inner = outer[i // ncol, i % ncol].subgridspec(1, 2, width_ratios=[3.1, 1.0],
                                                       wspace=0.06)
        ax = fig.add_subplot(inner[0]); axd = fig.add_subplot(inner[1], sharey=ax)
        draw(ax, axd, data[sid], ylabel=(i % ncol == 0), legend=(i == 0))
        ax.set_title(f"({chr(97+i)}) {sid.split('__')[0]} {data[sid]['sg_symbol']}",
                     loc="left")
    fig.savefig(os.path.join(OUT_DIR, "phonon_bands_panel.pdf"))
    fig.savefig(os.path.join(OUT_DIR, "phonon_bands_panel.png"), dpi=600)
    plt.close(fig)

cdf = pd.DataFrame(rows)
cdf.to_csv(os.path.join(OUT_DIR, "phonon_figure_audit.csv"), index=False)
print(f"\n{len(rows)}/{len(COMPOUNDS)} figures -> {OUT_DIR}/")
print(cdf.to_string(index=False))
print(f"\nacoustic residual max {cdf.acoustic_residual_THz.max():.2e} THz | "
      f"PhDOS sum-rule max deviation "
      f"{(cdf.dos_integral - cdf.dos_expected_3N).abs().max():.4f} states")

Saving the figures and readme

In [ ]:
import shutil

# Define the folder name and target ZIP file name
folder_name = "/content/figs_phonon"
zip_filename = "figs_phonon"  # Output will be figs_phonon.zip

# Create the ZIP archive
shutil.make_archive(zip_filename, "zip", folder_name)

print(f"Successfully zipped {folder_name} into {zip_filename}.zip!")